# Viewing silver Parquet exports

Silver files live under:

```
data/silver/locationid=<ID>/year=<YYYY>/<timestamp>_qnxhe_<uuid>
```

They are **Apache Parquet** files (magic bytes `PAR1`). Use **pandas**, **PyArrow**, or **DuckDB** — not a text editor.

In [ ]:
from pathlib import Path

import pandas as pd

SILVER_ROOT = Path("../data/silver")
exports = sorted(
    p for p in SILVER_ROOT.rglob("*") if p.is_file() and not p.name.startswith("_")
)
assert exports, f"No silver exports under {SILVER_ROOT.resolve()}"

DATA_FILE = exports[0]
print(f"Using: {DATA_FILE.relative_to(SILVER_ROOT.parent)}")
print(f"File size: {DATA_FILE.stat().st_size:,} bytes")
print(f"Magic bytes: {DATA_FILE.read_bytes()[:4]!r}  →  Parquet")

## 1. Load with pandas

In [ ]:
df = pd.read_parquet(DATA_FILE)

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
print(f"Location: {df['location'].iloc[0]}")
print(f"Date range: {df['datetime'].min()} → {df['datetime'].max()}")
print(f"Parameters: {sorted(df['parameter'].unique())}")

df.head(10)

## 2. Schema and summary stats

In [ ]:
df.info()
print()
df.groupby("parameter").agg(
    rows=("value", "count"),
    unit=("unit", "first"),
    min=("value", "min"),
    max=("value", "max"),
    mean=("value", "mean"),
)

## 3. Parquet metadata (PyArrow)

In [ ]:
import pyarrow.parquet as pq

table = pq.read_table(DATA_FILE)
meta = pq.read_metadata(DATA_FILE)

print(f"Row groups: {meta.num_row_groups}")
print(f"Rows: {meta.num_rows:,}")
print(f"Schema:\n{table.schema}")

## 4. Quick visual — PM2.5 over time

In [ ]:
import matplotlib.pyplot as plt

pm25 = df[df["parameter"] == "pm25"].sort_values("datetime")

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(pm25["datetime"], pm25["value"], linewidth=0.8)
ax.set_title(f"PM2.5 — {pm25['location'].iloc[0]}")
ax.set_xlabel("datetime")
ax.set_ylabel(pm25["unit"].iloc[0])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Other ways to view

| Tool | How |
|---|---|
| **This notebook** | `pd.read_parquet(path)` |
| **All silver exports** | `pd.read_parquet("../data/silver/locationid=*/year=*/*")` (needs pyarrow ≥14) |
| **DuckDB** | `SELECT * FROM read_parquet('data/silver/locationid=1544061/year=2026/*') LIMIT 10;` |

Build silver from bronze:

```bash
python -m pipelines conform
```